# ЛР № 4. Витрина почасового спроса на велопрокат Сеула

**Выбранный вопрос** (основной из [Project Brief](../docs/project_brief.md)):
как различается почасовой спрос на велосипеды по сезону, часу суток и типу дня
(рабочий, выходной, праздничный)?

**Метрики вопроса:**

| Метрика | Правило расчёта | Единица |
|---|---|---|
| Число наблюдений | `count(*)` по строкам группы | Часы |
| Суммарный спрос | `sum(rented_bike_count)` по группе | Аренды |
| Средний спрос за час | `avg(rented_bike_count)` по группе | Аренды/час |
| Медианный спрос за час | `median(rented_bike_count)` по группе | Аренды/час |

**Срезы:** `season`, `hour`, `day_type` (рабочий, выходной, праздничный), `month`,
`day_of_week`. Дополнительная мера для уточняющего вопроса о погоде — `temperature_c`.

Продолжаем [решения ЛР № 3](../docs/data_quality_report.md). Модель и происхождение полей
описаны в [docs/mart.md](../docs/mart.md). Полный расчёт метрик и содержательные выводы
остаются следующему этапу: здесь готовится и проверяется сама витрина.

## 1. Параметры и вход

Логическая дата берётся из пути подготовленного `staging`, а не из даты запуска.
`source_ingested_on=2026-11-12` обозначает загрузку исходных данных и **не** является
временем наблюдения: время события хранится в поле `observed_on`.

Ноутбук запускается из корня репозитория или каталога `notebooks` с ядром проектного
`.venv`. Стенд должен работать, настройки S3 читаются из `.env`. Ingestion и ЛР № 3
повторно выполнять не нужно.

In [1]:
from datetime import date

DATASET_SLUG = "seoul_bike"
SOURCE_INGESTED_ON = "2026-11-12"
MART_NAME = "hourly_demand"

date.fromisoformat(SOURCE_INGESTED_ON)

PREFIX = f"{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}"
STAGING_PATH = f"s3://staging/{PREFIX}/staging.parquet"
MART_KEY = f"{PREFIX}/{MART_NAME}.parquet"
MART_PATH = f"s3://mart/{MART_KEY}"

print(f"Вход:  {STAGING_PATH}")
print(f"Выход: {MART_PATH}")

Вход:  s3://staging/seoul_bike/source_ingested_on=2026-11-12/staging.parquet
Выход: s3://mart/seoul_bike/source_ingested_on=2026-11-12/hourly_demand.parquet


In [2]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required_settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing_settings)}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402


def literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


check_results = []


def check(label, actual, expected):
    passed = actual == expected
    check_results.append((label, actual, expected, passed))
    assert passed, f"{label}: получено {actual!r}, ожидалось {expected!r}"


connection = duckdb.connect()
configure_s3(connection)
endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
source_before = client.head_object(Bucket="staging", Key=f"{PREFIX}/staging.parquet")

### Проверка входа перед сборкой

`hive_partitioning = false` отключает автоматическое добавление полей из каталогов:
проверяется физическая схема файла. Логическая дата уже задана его путём.

In [3]:
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging AS
    SELECT * FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false)
""")
source_schema = {row[0]: row[1] for row in connection.execute("DESCRIBE staging").fetchall()}
required_fields = {
    "observed_on", "hour", "rented_bike_count", "season", "holiday",
    "functioning_day", "service_closed", "temperature_c", "rainfall_mm", "snowfall_cm",
    "quality_humidity_zero_is_suspicious_violated",
    "quality_snowfall_requires_cold_violated",
}
check("Наличие нужных полей staging", required_fields - source_schema.keys(), set())

source_counts = connection.execute("""
    SELECT
        count(*) AS rows_count,
        count(*) FILTER (WHERE rented_bike_count IS NULL) AS null_demand,
        count(*) FILTER (
            WHERE quality_humidity_zero_is_suspicious_violated
        ) AS flagged_humidity,
        count(*) FILTER (
            WHERE quality_snowfall_requires_cold_violated
        ) AS flagged_snowfall,
        count(*) FILTER (WHERE service_closed) AS closed_hours,
        count(*) FILTER (
            WHERE observed_on IS NULL OR hour IS NULL OR rented_bike_count IS NULL
               OR season IS NULL OR holiday IS NULL OR functioning_day IS NULL
               OR hour NOT BETWEEN 0 AND 23 OR rented_bike_count < 0
               OR (service_closed AND rented_bike_count <> 0)
        ) AS critical_violations
    FROM staging
""").fetchone()
(
    staging_rows,
    source_null_demand,
    source_flagged_humidity,
    source_flagged_snowfall,
    source_closed_hours,
    source_critical,
) = source_counts
check("Критические нарушения ЛР № 3 на входе", source_critical, 0)
check("Повторы ключа дата + час на входе", connection.execute("""
    SELECT count(*) FROM (
        SELECT observed_on, hour FROM staging GROUP BY 1, 2 HAVING count(*) > 1
    )
""").fetchone()[0], 0)
show_table(
    ["Строк staging", "NULL спроса", "Флаг влажности", "Флаг снега",
     "Часов закрытого сервиса", "Критические нарушения"],
    [source_counts],
)
show_table(
    ["Нужное поле staging", "Тип"],
    [(name, source_schema[name]) for name in sorted(required_fields)],
)

Строк staging,NULL спроса,Флаг влажности,Флаг снега,Часов закрытого сервиса,Критические нарушения
8760,0,17,14,295,0


Нужное поле staging,Тип
functioning_day,BOOLEAN
holiday,VARCHAR
hour,BIGINT
observed_on,DATE
quality_humidity_zero_is_suspicious_violated,BOOLEAN
quality_snowfall_requires_cold_violated,BOOLEAN
rainfall_mm,DOUBLE
rented_bike_count,BIGINT
season,VARCHAR
service_closed,BOOLEAN


## 2. Гранулярность, ключ и меры

**Одна строка `hourly_demand` представляет один час работы системы велопроката
в определённую дату** — то же, что и строка `staging`. Исходные данные уже агрегированы
по часам, поэтому уровень детализации сохраняется: `count(*)` считает часы наблюдений,
а число аренд хранится отдельной мерой `rented_bike_count`.

**Ключ.** Надёжного бизнес-ключа в источнике нет: отдельная поездка в данных не видна.
Сочетание `observed_on` + `hour` уникально (проверено на входе), и это подтверждает
заявленную гранулярность. Однако уникальность часа наблюдения **не** доказывает
уникальность реальных событий: разные поездки одного часа неразличимы по построению
набора. Поэтому технический идентификатор строки не создаётся, `DISTINCT` для
искусственной уникальности не применяется, а кратность строк сохраняется.

| Мера | Единица | Использование в вопросе | Допустимость агрегации |
|---|---|---|---|
| `rented_bike_count` | Аренды за час | Основная метрика: сумма и среднее по группам | Суммируется по непересекающимся группам одного снимка. Средние групп нельзя усреднять без учёта размера групп |
| `hour` | Час суток 0–23 | Срез: сравнение по часам | Не мера, а признак группировки |
| `temperature_c` | °C | Уточняющий вопрос о погоде | Не суммируется как итог; для связи используется с оговорками таблицы выше |

Меры нельзя складывать между разными снимками источника: один и тот же час может
попасть в два снимка. Набор содержит один непрерывный период, поэтому это ограничение
касается будущих загрузок.

## 3. Выбор структуры и сопоставление полей

Выбираем **одну широкую таблицу**. Причины:

- Все необходимые для вопроса атрибуты — сезон, тип дня, календарные признаки и погодные
  показатели — уже находятся в одной строке `staging` с согласованной гранулярностью.
- Отдельных справочников у этого набора нет: `season` и `holiday` — это короткие
  перечисления прямо в данных, а не таблицы измерений. Выделять их в отдельные объекты
  `mart` значило бы добавить соединение без пользы для вопроса.
- Одна широкая таблица сохраняет исходную кратность строк, что упрощает проверку баланса
  с `staging`.

**Производные признаки:**

- `day_type` — тип дня с приоритетом праздника: `праздник`, если `holiday = 'Holiday'`;
  иначе `выходной`, если ISO-день недели 6 или 7; иначе `рабочий`. Приоритет нужен потому,
  что часть праздников выпадает на субботу или воскресенье — таких дней в наборе четыре.
  Без приоритета они попали бы в две группы сразу.
- `month`, `day_of_week`, `week_of_year` — календарные признаки из `observed_on`
  для будущих срезов. `day_of_week` берётся как ISO-номер: 1 — понедельник, 7 — воскресенье.
- `is_open` — обратный признак к `service_closed`, чтобы фильтр открытых часов читался
  естественно.

In [4]:
connection.execute("""
    CREATE OR REPLACE TEMP TABLE hourly_demand AS
    SELECT
        s.observed_on                                        AS observed_on,
        s.hour                                               AS hour,
        s.rented_bike_count                                  AS rented_bike_count,
        s.season                                             AS season,
        s.holiday                                            AS holiday,
        CASE
            WHEN s.holiday = 'Holiday' THEN 'праздник'
            WHEN EXTRACT(isodow FROM s.observed_on) IN (6, 7) THEN 'выходной'
            ELSE 'рабочий'
        END                                                  AS day_type,
        CAST(EXTRACT(month FROM s.observed_on) AS INTEGER)    AS month,
        CAST(EXTRACT(isodow FROM s.observed_on) AS INTEGER)   AS day_of_week,
        CAST(EXTRACT(week FROM s.observed_on) AS INTEGER)     AS week_of_year,
        s.functioning_day                                    AS functioning_day,
        s.service_closed                                     AS service_closed,
        NOT s.service_closed                                 AS is_open,
        s.temperature_c                                      AS temperature_c,
        s.humidity_pct                                       AS humidity_pct,
        s.wind_speed_ms                                      AS wind_speed_ms,
        s.rainfall_mm                                        AS rainfall_mm,
        s.snowfall_cm                                        AS snowfall_cm,
        s.quality_humidity_zero_is_suspicious_violated
            AS humidity_zero_flag,
        s.quality_snowfall_requires_cold_violated
            AS warm_snowfall_flag
    FROM staging AS s
""")

prepared_rows = connection.execute("SELECT count(*) FROM hourly_demand").fetchone()[0]
check("Сохранение строк при преобразовании", prepared_rows, staging_rows)
show_table(
    ["Дата", "Час", "Аренд", "Сезон", "Тип дня", "Месяц", "ISO-день", "Открыт", "Темп."],
    connection.execute("""
        SELECT observed_on, hour, rented_bike_count, season, day_type,
               month, day_of_week, is_open, temperature_c
        FROM hourly_demand
        ORDER BY observed_on, hour LIMIT 5
    """).fetchall(),
)

Дата,Час,Аренд,Сезон,Тип дня,Месяц,ISO-день,Открыт,Темп.
2017-12-01,0,254,Winter,рабочий,12,5,True,-5.2
2017-12-01,1,204,Winter,рабочий,12,5,True,-5.5
2017-12-01,2,173,Winter,рабочий,12,5,True,-6.0
2017-12-01,3,107,Winter,рабочий,12,5,True,-6.2
2017-12-01,4,78,Winter,рабочий,12,5,True,-6.0


### Фильтрация не применяется

Все строки `staging` включаются в витрину: **исключённых строк нет**. Это сознательное
решение, продолжающее ЛР № 3:

- Часы остановки сервиса (295 строк) остаются с признаком `service_closed`. В ЛР № 3 они
  признаны **не** ошибкой данных, а реальным состоянием системы. Удалять их здесь значило
  бы принять новое решение о качестве и потерять сведения о доступности сервиса.
- Обратите внимание: удаление этих строк подняло бы средний спрос с 704.6 до 729.16
  аренды в час — то есть молча изменило бы метрику. Поэтому строки сохранены,
  а фильтр `is_open` остаётся выбором будущего расчёта.
- Строки с флагами влажности (17) и снега (14) также сохранены: сомнительно только
  погодное значение, а число аренд в этих часах достоверно.

Баланс: `count(входных) = count(сохранённых) + count(исключённых)` → 8760 = 8760 + 0.

## 4. Запись и повторное чтение S3

Используем способ ЛР № 3: удаляем только целевой объект `hourly_demand.parquet`
и записываем его заново. Повторный запуск заменяет результат, не добавляя копий строк.
Остальные объекты S3 не затрагиваются.

После записи открываем новое соединение DuckDB и читаем именно объект S3. Автоматическое
добавление полей из каталогов отключено: проверяется физическая схема файла.

In [5]:
assert MART_PATH == f"s3://mart/{PREFIX}/{MART_NAME}.parquet"
assert MART_KEY == f"{PREFIX}/{MART_NAME}.parquet"
client.head_bucket(Bucket="mart")
client.delete_object(Bucket="mart", Key=MART_KEY)
connection.execute(f"""
    COPY hourly_demand TO {literal(MART_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")

published = duckdb.connect()
configure_s3(published)
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW published_mart AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")
output_head = client.head_object(Bucket="mart", Key=MART_KEY)
show_table(["Объект", "Размер, байт"], [(MART_PATH, output_head["ContentLength"])])

Объект,"Размер, байт"
s3://mart/seoul_bike/source_ingested_on=2026-11-12/hourly_demand.parquet,84573


<a id="checks"></a>
## 5. Проверки сохранённой витрины

Все запросы ниже читают записанный Parquet на S3. Ожидания следуют из модели и входного
`staging`. При нарушении любого условия `assert` останавливает выполнение. Сводка в конце
показывает фактическое и ожидаемое значение каждой проверки.

### Количество и схема

In [6]:
mart_rows = published.execute("SELECT count(*) FROM published_mart").fetchone()[0]
excluded_rows = staging_rows - prepared_rows
check("Исключённые строки", excluded_rows, 0)
check("Баланс входных и сохранённых строк", staging_rows, mart_rows + excluded_rows)
check("Количество после чтения S3", mart_rows, prepared_rows)

expected_schema = [
    ("observed_on", "DATE"),
    ("hour", "BIGINT"),
    ("rented_bike_count", "BIGINT"),
    ("season", "VARCHAR"),
    ("holiday", "VARCHAR"),
    ("day_type", "VARCHAR"),
    ("month", "INTEGER"),
    ("day_of_week", "INTEGER"),
    ("week_of_year", "INTEGER"),
    ("functioning_day", "BOOLEAN"),
    ("service_closed", "BOOLEAN"),
    ("is_open", "BOOLEAN"),
    ("temperature_c", "DOUBLE"),
    ("humidity_pct", "BIGINT"),
    ("wind_speed_ms", "DOUBLE"),
    ("rainfall_mm", "DOUBLE"),
    ("snowfall_cm", "DOUBLE"),
    ("humidity_zero_flag", "BOOLEAN"),
    ("warm_snowfall_flag", "BOOLEAN"),
]
actual_schema = [row[:2] for row in published.execute("DESCRIBE published_mart").fetchall()]
check("Имена, порядок и типы 19 полей", actual_schema, expected_schema)
show_table(["Staging", "Подготовлено", "Прочитано из mart", "Исключено"], [
    (staging_rows, prepared_rows, mart_rows, excluded_rows)
])
show_table(["Поле", "Тип"], actual_schema)

Staging,Подготовлено,Прочитано из mart,Исключено
8760,8760,8760,0


Поле,Тип
observed_on,DATE
hour,BIGINT
rented_bike_count,BIGINT
season,VARCHAR
holiday,VARCHAR
day_type,VARCHAR
month,INTEGER
day_of_week,INTEGER
week_of_year,INTEGER
functioning_day,BOOLEAN


### Ключ и обязательные поля

Ключ `observed_on` + `hour` выведен из гранулярности набора. Проверяем отсутствие пропусков
в обоих компонентах и отсутствие повторов. Без даты и часа теряется смысл строки.

In [7]:
null_keys = published.execute("""
    SELECT count(*) FROM published_mart WHERE observed_on IS NULL OR hour IS NULL
""").fetchone()[0]
duplicate_keys = published.execute("""
    SELECT count(*) FROM (
        SELECT observed_on, hour FROM published_mart
        GROUP BY 1, 2 HAVING count(*) > 1
    )
""").fetchone()[0]
check("Пропуски в компонентах ключа", null_keys, 0)
check("Повторы ключа в витрине", duplicate_keys, 0)

null_counts = published.execute("""
    SELECT
        count(*) FILTER (WHERE rented_bike_count IS NULL),
        count(*) FILTER (WHERE season IS NULL),
        count(*) FILTER (WHERE holiday IS NULL),
        count(*) FILTER (WHERE day_type IS NULL),
        count(*) FILTER (WHERE month IS NULL),
        count(*) FILTER (WHERE day_of_week IS NULL),
        count(*) FILTER (WHERE week_of_year IS NULL),
        count(*) FILTER (WHERE functioning_day IS NULL),
        count(*) FILTER (WHERE service_closed IS NULL),
        count(*) FILTER (WHERE is_open IS NULL),
        count(*) FILTER (WHERE humidity_zero_flag IS NULL),
        count(*) FILTER (WHERE warm_snowfall_flag IS NULL),
        count(*) FILTER (WHERE temperature_c IS NULL)
    FROM published_mart
""").fetchone()
required_names = [
    "rented_bike_count", "season", "holiday", "day_type", "month", "day_of_week",
    "week_of_year", "functioning_day", "service_closed", "is_open",
    "humidity_zero_flag", "warm_snowfall_flag", "temperature_c",
]
for name, value in zip(required_names, null_counts):
    check(f"NULL в обязательном поле {name}", value, 0)
show_table(["Поле", "NULL"], list(zip(required_names, null_counts)))
show_table(["Проверка ключа", "Значение"], [
    ("Пропуски observed_on или hour", null_keys),
    ("Повторы пары дата + час", duplicate_keys),
])

Поле,NULL
rented_bike_count,0
season,0
holiday,0
day_type,0
month,0
day_of_week,0
week_of_year,0
functioning_day,0
service_closed,0
is_open,0


Проверка ключа,Значение
Пропуски observed_on или hour,0
Повторы пары дата + час,0


### Меры и вычисленные поля

Проверяем диапазоны, которые следуют из определений полей: час 0–23, месяц 1–12,
ISO-день недели 1–7, номер недели 1–53. Значения мер сравниваем со `staging`:
витрина не должна ни отсекать, ни изменять их.

In [8]:
measure_profile = published.execute("""
    SELECT
        min(rented_bike_count), max(rented_bike_count), sum(rented_bike_count),
        min(temperature_c), max(temperature_c),
        count(*) FILTER (WHERE rented_bike_count < 0)
    FROM published_mart
""").fetchone()
source_measure = connection.execute("""
    SELECT min(rented_bike_count), max(rented_bike_count), sum(rented_bike_count),
           min(temperature_c), max(temperature_c)
    FROM staging
""").fetchone()
check("Меры совпадают со staging", measure_profile[:5], source_measure)
check("Отрицательный спрос в витрине", measure_profile[5], 0)

derived_violations = published.execute("""
    SELECT count(*) FROM published_mart
    WHERE hour NOT BETWEEN 0 AND 23
       OR month NOT BETWEEN 1 AND 12
       OR day_of_week NOT BETWEEN 1 AND 7
       OR week_of_year NOT BETWEEN 1 AND 53
       OR month IS DISTINCT FROM EXTRACT(month FROM observed_on)
       OR day_of_week IS DISTINCT FROM EXTRACT(isodow FROM observed_on)
       OR week_of_year IS DISTINCT FROM EXTRACT(week FROM observed_on)
       OR is_open IS DISTINCT FROM NOT service_closed
       OR service_closed IS DISTINCT FROM NOT functioning_day
       OR day_type IS DISTINCT FROM (
            CASE WHEN holiday = 'Holiday' THEN 'праздник'
                 WHEN EXTRACT(isodow FROM observed_on) IN (6, 7) THEN 'выходной'
                 ELSE 'рабочий' END)
""").fetchone()[0]
check("Нарушения вычисленных полей", derived_violations, 0)
day_type_values = published.execute("""
    SELECT DISTINCT day_type FROM published_mart ORDER BY 1
""").fetchall()
check("Набор значений day_type", [r[0] for r in day_type_values],
      ["выходной", "праздник", "рабочий"])
check("Сезон принимает четыре значения",
      published.execute("SELECT count(DISTINCT season) FROM published_mart").fetchone()[0], 4)
show_table(["Мера", "Минимум", "Максимум", "Сумма", "Единица"], [
    ("rented_bike_count", measure_profile[0], measure_profile[1], measure_profile[2],
     "Аренды за час"),
    ("temperature_c", measure_profile[3], measure_profile[4], "не суммируется", "°C"),
])
show_table(["Проверка", "Нарушений"], [
    ("Вычисленные поля и диапазоны", derived_violations),
    ("Отрицательный спрос", measure_profile[5]),
])
show_table(["day_type", "Число значений"], [(v[0], "") for v in day_type_values])

Мера,Минимум,Максимум,Сумма,Единица
rented_bike_count,0,3556,6172314,Аренды за час
temperature_c,-17.8,39.4,не суммируется,°C


Проверка,Нарушений
Вычисленные поля и диапазоны,0
Отрицательный спрос,0


day_type,Число значений
выходной,
праздник,
рабочий,


### Флаги качества из ЛР № 3

Флаги переносятся без изменения значений: строка, помеченная в ЛР № 3, должна остаться
помеченной в витрине, а её измеренное значение — сохраниться, а не заменяться нулём
или `NULL`. Сверяем количество помеченных строк со `staging` и проверяем, что флаги
соответствуют самим значениям.

In [9]:
flag_profile = published.execute("""
    SELECT
        count(*) FILTER (WHERE humidity_zero_flag) AS humidity_flagged,
        count(*) FILTER (WHERE warm_snowfall_flag) AS snowfall_flagged,
        count(*) FILTER (WHERE service_closed) AS closed_rows,
        count(*) FILTER (
            WHERE humidity_zero_flag IS DISTINCT FROM (humidity_pct = 0)
        ) AS humidity_mismatches,
        count(*) FILTER (
            WHERE warm_snowfall_flag
                  IS DISTINCT FROM (snowfall_cm > 0 AND temperature_c >= 5)
        ) AS snowfall_mismatches
    FROM published_mart
""").fetchone()
check("Флаг влажности относительно staging", flag_profile[0], source_flagged_humidity)
check("Флаг снега относительно staging", flag_profile[1], source_flagged_snowfall)
check("Часы закрытого сервиса относительно staging", flag_profile[2], source_closed_hours)
check("Несоответствия флага влажности значению", flag_profile[3], 0)
check("Несоответствия флага снега значению", flag_profile[4], 0)

flagged_values_kept = published.execute("""
    SELECT count(*) FROM published_mart
    WHERE (humidity_zero_flag AND humidity_pct IS DISTINCT FROM 0)
       OR (warm_snowfall_flag AND (snowfall_cm IS NULL OR snowfall_cm <= 0))
       OR (service_closed AND rented_bike_count IS DISTINCT FROM 0)
""").fetchone()[0]
check("Значения помеченных строк не подменены", flagged_values_kept, 0)
show_table(
    ["Флаг влажности", "Флаг снега", "Закрытый сервис", "Несоответствия влажности",
     "Несоответствия снега"],
    [flag_profile],
)

Флаг влажности,Флаг снега,Закрытый сервис,Несоответствия влажности,Несоответствия снега
17,14,295,0,0


### Сохранение значений и кратности строк

Бизнес-ключа нет, поэтому сверяем все 19 полей витрины с ожидаемой проекцией `staging`
через `EXCEPT ALL` в обе стороны. `EXCEPT ALL` учитывает кратность: совпавшие
аналитические проекции не должны исчезать.

In [10]:
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW expected_mart AS
    SELECT
        observed_on,
        hour,
        rented_bike_count,
        season,
        holiday,
        CASE
            WHEN holiday = 'Holiday' THEN 'праздник'
            WHEN EXTRACT(isodow FROM observed_on) IN (6, 7) THEN 'выходной'
            ELSE 'рабочий'
        END AS day_type,
        CAST(EXTRACT(month FROM observed_on) AS INTEGER) AS month,
        CAST(EXTRACT(isodow FROM observed_on) AS INTEGER) AS day_of_week,
        CAST(EXTRACT(week FROM observed_on) AS INTEGER) AS week_of_year,
        functioning_day,
        service_closed,
        NOT service_closed AS is_open,
        temperature_c,
        humidity_pct,
        wind_speed_ms,
        rainfall_mm,
        snowfall_cm,
        quality_humidity_zero_is_suspicious_violated AS humidity_zero_flag,
        quality_snowfall_requires_cold_violated AS warm_snowfall_flag
    FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false)
""")
content_difference = published.execute("""
    SELECT 'Отсутствуют в mart' AS direction, count(*) AS differing_rows
    FROM (SELECT * FROM expected_mart EXCEPT ALL SELECT * FROM published_mart)
    UNION ALL
    SELECT 'Лишние в mart', count(*)
    FROM (SELECT * FROM published_mart EXCEPT ALL SELECT * FROM expected_mart)
""").fetchall()
for direction, differing_rows in content_difference:
    check(direction, differing_rows, 0)
show_table(["Направление сверки", "Различающихся строк"], content_difference)

Направление сверки,Различающихся строк
Отсутствуют в mart,0
Лишние в mart,0


### Пригодность для выбранного вопроса

Контрольный запрос группирует часы по сезону и типу дня — это срезы выбранного вопроса.
Сумма числа часов по всем группам должна равняться общему числу строк. Полный расчёт
метрик, сравнения и содержательные выводы остаются следующему этапу.

In [11]:
group_counts = published.execute("""
    SELECT season, day_type, count(*) AS hours, sum(rented_bike_count) AS rented
    FROM published_mart
    GROUP BY 1, 2 ORDER BY 1, 2
""").fetchall()
check("Баланс контрольной группировки",
      sum(row[2] for row in group_counts), mart_rows)
check("Число сочетаний сезон x тип дня",
      len(group_counts), 4 * 3)

open_summary = published.execute("""
    SELECT
        count(*) AS all_hours,
        count(*) FILTER (WHERE is_open) AS open_hours,
        round(avg(rented_bike_count), 2) AS avg_all,
        round(avg(rented_bike_count) FILTER (WHERE is_open), 2) AS avg_open
    FROM published_mart
""").fetchone()
show_table(["Сезон", "Тип дня", "Часов", "Аренд за период"], group_counts)
show_table(["Всего часов", "Открытых часов", "Средний спрос (все)",
            "Средний спрос (открытые)"], [open_summary])

Сезон,Тип дня,Часов,Аренд за период
Autumn,выходной,600,480729
Autumn,праздник,120,91018
Autumn,рабочий,1464,1218255
Spring,выходной,624,425931
Spring,праздник,72,45742
Spring,рабочий,1512,1140236
Summer,выходной,624,620759
Summer,праздник,48,49063
Summer,рабочий,1536,1613412
Winter,выходной,552,109652


Всего часов,Открытых часов,Средний спрос (все),Средний спрос (открытые)
8760,8465,704.6,729.16


### Контрольная сводка

Дополнительно проверяем, что метаданные входного объекта не изменились за время
выполнения. Все проверки относятся к конкретному снимку и не доказывают полноту
или достоверность сведений о реальном спросе.

In [12]:
source_after = client.head_object(Bucket="staging", Key=f"{PREFIX}/staging.parquet")
check("Входной объект не изменён", (
    source_after["ETag"], source_after["ContentLength"], source_after["LastModified"]
), (
    source_before["ETag"], source_before["ContentLength"], source_before["LastModified"]
))
show_table(["Проверка", "Получено", "Ожидается", "Итог"], [
    (name, actual, expected, "Пройдена" if passed else "Нарушение")
    for name, actual, expected, passed in check_results
])
show_table(["Проверок", "Пройдено", "Строк mart", "Флагов сохранено"], [
    (len(check_results), sum(passed for _, _, _, passed in check_results), mart_rows,
     flag_profile[0] + flag_profile[1]),
])
published.close()
connection.close()

Проверка,Получено,Ожидается,Итог
Наличие нужных полей staging,set(),set(),Пройдена
Критические нарушения ЛР № 3 на входе,0,0,Пройдена
Повторы ключа дата + час на входе,0,0,Пройдена
Сохранение строк при преобразовании,8760,8760,Пройдена
Исключённые строки,0,0,Пройдена
Баланс входных и сохранённых строк,8760,8760,Пройдена
Количество после чтения S3,8760,8760,Пройдена
"Имена, порядок и типы 19 полей","[('observed_on', 'DATE'), ('hour', 'BIGINT'), ('rented_bike_count', 'BIGINT'), ('season', 'VARCHAR'), ('holiday', 'VARCHAR'), ('day_type', 'VARCHAR'), ('month', 'INTEGER'), ('day_of_week', 'INTEGER'), ('week_of_year', 'INTEGER'), ('functioning_day', 'BOOLEAN'), ('service_closed', 'BOOLEAN'), ('is_open', 'BOOLEAN'), ('temperature_c', 'DOUBLE'), ('humidity_pct', 'BIGINT'), ('wind_speed_ms', 'DOUBLE'), ('rainfall_mm', 'DOUBLE'), ('snowfall_cm', 'DOUBLE'), ('humidity_zero_flag', 'BOOLEAN'), ('warm_snowfall_flag', 'BOOLEAN')]","[('observed_on', 'DATE'), ('hour', 'BIGINT'), ('rented_bike_count', 'BIGINT'), ('season', 'VARCHAR'), ('holiday', 'VARCHAR'), ('day_type', 'VARCHAR'), ('month', 'INTEGER'), ('day_of_week', 'INTEGER'), ('week_of_year', 'INTEGER'), ('functioning_day', 'BOOLEAN'), ('service_closed', 'BOOLEAN'), ('is_open', 'BOOLEAN'), ('temperature_c', 'DOUBLE'), ('humidity_pct', 'BIGINT'), ('wind_speed_ms', 'DOUBLE'), ('rainfall_mm', 'DOUBLE'), ('snowfall_cm', 'DOUBLE'), ('humidity_zero_flag', 'BOOLEAN'), ('warm_snowfall_flag', 'BOOLEAN')]",Пройдена
Пропуски в компонентах ключа,0,0,Пройдена
Повторы ключа в витрине,0,0,Пройдена


Проверок,Пройдено,Строк mart,Флагов сохранено
39,39,8760,31


<a id="result"></a>
## 6. Оформление результата

Модель, происхождение всех выходных полей, краткий итог проверок и ограничения описаны
в [docs/mart.md](../docs/mart.md).

Итог сборки: все 8760 строк `staging` сохранены в витрине, исключённых строк нет.
Гранулярность «один час одних суток» и кратность строк сохранены. В выходном файле
19 полей; сохранены признаки ЛР № 3 (17 строк с флагом влажности, 14 — со снегом)
и добавлен признак `service_closed` (295 часов).

Витрина сохранена и прочитана из S3:

```text
s3://mart/seoul_bike/source_ingested_on=2026-11-12/hourly_demand.parquet
```

Ограничения: бизнес-ключа нет, поэтому доказать уникальность реальных поездок нельзя;
данные агрегированы по часам и описывают один город за один непрерывный год; часы
остановки сервиса остаются в витрине и учитываются отдельно по признаку `is_open`;
погодные показатели — наблюдения метеослужбы, а не условия в точке аренды.